In [0]:
dbutils.widgets.text("catalog","sj20030614")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_station AS
SELECT * FROM {catalog}.silver.stations;             
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_charger AS
SELECT charger_id,station_id,connector_type,max_kw,is_active,commissioned_on FROM {catalog}.silver.chargers;             
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_member AS
SELECT
  CAST(ROW_NUMBER() OVER (ORDER BY member_id, effective_from) AS BIGINT) AS member_sk,
  *
FROM {catalog}.silver.members;
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_date AS
SELECT
  CAST(DATE_FORMAT(d.cal_date, 'yyyyMMdd') AS INT)         AS date_key,
  d.cal_date                                            AS calendar_date,
  DATE_FORMAT(d.cal_date, 'EEEE') AS day_name,
  WEEKOFYEAR(d.cal_date) AS iso_week,
  CASE WHEN DAYOFWEEK(d.cal_date) IN (1, 7)
       THEN TRUE ELSE FALSE END AS is_weekend
FROM (
  SELECT explode(sequence(
           to_date('2026-04-01'),
           to_date('2026-04-30'),
           interval 1 day)) AS cal_date
) d;
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.fact_sessions AS
WITH fault_counts AS (
    SELECT session_id, COUNT(*) AS fault_count
    FROM {catalog}.silver.session_faults
    GROUP BY session_id
),
base AS (
    SELECT
        s.session_id,
        d.date_key,
        s.charger_id,
        c.station_id,
        s.member_id,
        s.status,
        s.time_band,
        s.energy_kwh,
        s.duration_minutes,
        s.idle_minutes,
        s.amount_paid,
        CASE WHEN upper(s.member_id) = 'GUEST' THEN NULL
             ELSE m.member_sk END AS member_sk,
        CASE WHEN upper(s.member_id) = 'GUEST' THEN 'GUEST'
             ELSE m.plan END AS plan_at_session,
        t.rate_per_kwh,
        t.free_idle_minutes,
        t.idle_fee_per_min
    FROM {catalog}.silver.sessions s
    LEFT JOIN {catalog}.gold.dim_date    d ON s.session_date = d.calendar_date
    LEFT JOIN {catalog}.gold.dim_charger c ON s.charger_id   = c.charger_id
    LEFT JOIN {catalog}.gold.dim_member  m
           ON upper(s.member_id) != 'GUEST'
          AND s.member_id = m.member_id
          AND s.session_date >= m.effective_from
          AND (s.session_date < m.effective_to OR m.effective_to IS NULL)
    LEFT JOIN {catalog}.silver.tariffs   t
           ON c.connector_type = t.connector_type
          AND s.time_band = t.time_band
),
billing AS (
    SELECT
        b.*,
        CASE WHEN status = 'COMPLETED' THEN
            CAST(ROUND(energy_kwh * rate_per_kwh, 2) AS DECIMAL(10,2))
        ELSE CAST(0.00 AS DECIMAL(10,2)) END AS energy_charge,
        CASE WHEN status = 'COMPLETED' THEN
            CAST(ROUND(
                ROUND(energy_kwh * rate_per_kwh, 2) *
                CASE plan_at_session
                    WHEN 'BASIC' THEN 0.00
                    WHEN 'PRO'   THEN 0.10
                    WHEN 'FLEET' THEN 0.15
                    ELSE 0.00
                END
            , 2) AS DECIMAL(10,2))
        ELSE CAST(0.00 AS DECIMAL(10,2)) END AS plan_discount,
        CASE WHEN status = 'COMPLETED' THEN
            CAST(ROUND(
                GREATEST(idle_minutes - free_idle_minutes, 0) * idle_fee_per_min
            , 2) AS DECIMAL(10,2))
        ELSE CAST(0.00 AS DECIMAL(10,2)) END AS idle_fee
    FROM base b
)
SELECT
    session_id,
    date_key,
    charger_id,
    station_id,
    member_id,
    status,
    time_band,
    energy_kwh,
    duration_minutes,
    idle_minutes,
    amount_paid,
    member_sk,
    plan_at_session,
    rate_per_kwh,
    energy_charge,
    plan_discount,
    idle_fee,
    CAST(energy_charge - plan_discount + idle_fee AS DECIMAL(10,2)) AS billed_amount,
    CASE WHEN status = 'COMPLETED' THEN
        CASE WHEN ABS(amount_paid - (energy_charge - plan_discount + idle_fee)) <= 0.50
             THEN 'OK' ELSE 'MISMATCH' END
    ELSE NULL END AS billing_status,
    COALESCE(f.fault_count, 0) AS fault_count
FROM billing
LEFT JOIN fault_counts f USING (session_id);
""")

print(f"gold.fact_sessions: {spark.table(f'{catalog}.gold.fact_sessions').count()} rows")

#gold.agg_daily_station

In [0]:
spark.sql("SET spark.sql.ansi.enabled = false")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.agg_daily_station AS
WITH date_spine AS (
    SELECT explode(sequence(
        (SELECT min(session_date) FROM {catalog}.silver.sessions),
        (SELECT max(session_date) FROM {catalog}.silver.sessions),
        interval 1 day
    )) AS session_date
),
station_grid AS (
    SELECT d.session_date, s.station_id, s.city
    FROM date_spine d
    CROSS JOIN (SELECT DISTINCT station_id, city FROM {catalog}.gold.dim_station) s
),
charger_counts AS (
    SELECT station_id, COUNT(*) AS charger_count
    FROM {catalog}.gold.dim_charger
    GROUP BY station_id
),
daily_agg AS (
    SELECT
        dt.calendar_date AS session_date,
        f.station_id,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_sessions,
        SUM(CASE WHEN f.status = 'FAILED'    THEN 1 ELSE 0 END) AS failed_sessions,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN f.energy_kwh   ELSE 0 END) AS energy_kwh,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN f.billed_amount ELSE 0 END) AS billed_amount,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN f.amount_paid  ELSE 0 END) AS amount_paid,
        SUM(CASE WHEN f.billing_status = 'MISMATCH' THEN 1 ELSE 0 END) AS mismatch_sessions,
        COALESCE(SUM(f.fault_count), 0) AS fault_count,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN f.duration_minutes ELSE 0 END) AS completed_duration
    FROM {catalog}.gold.fact_sessions f
    JOIN {catalog}.gold.dim_date dt ON f.date_key = dt.date_key
    GROUP BY dt.calendar_date, f.station_id
)
SELECT
    g.session_date,
    g.station_id,
    g.city,
    COALESCE(a.completed_sessions, 0)  AS completed_sessions,
    COALESCE(a.failed_sessions, 0)      AS failed_sessions,
    COALESCE(a.energy_kwh, 0)           AS energy_kwh,
    COALESCE(a.billed_amount, 0)        AS billed_amount,
    COALESCE(a.amount_paid, 0)          AS amount_paid,
    COALESCE(a.mismatch_sessions, 0)    AS mismatch_sessions,
    COALESCE(a.fault_count, 0)          AS fault_count,
    CAST(ROUND(
        CAST(100.0 * COALESCE(a.completed_duration, 0) / (c.charger_count * 1440) AS DOUBLE)
    , 2) AS DOUBLE) AS utilisation_pct
FROM station_grid g
LEFT JOIN daily_agg      a ON g.session_date = a.session_date AND g.station_id = a.station_id
LEFT JOIN charger_counts c ON g.station_id   = c.station_id;
""")

print(f"gold.agg_daily_station: {spark.table(f'{catalog}.gold.agg_daily_station').count()} rows")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.v_kpi_summary AS
WITH agg AS (
    SELECT
        COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END)        AS completed_sessions,
        COUNT(CASE WHEN status = 'FAILED'    THEN 1 END)        AS failed_sessions,
        COUNT(CASE WHEN status IN ('COMPLETED','FAILED') THEN 1 END) AS total_cf,
        SUM(CASE WHEN status = 'COMPLETED' THEN energy_kwh   ELSE 0 END) AS total_energy_kwh,
        SUM(CASE WHEN status = 'COMPLETED' THEN billed_amount ELSE 0 END) AS total_billed,
        SUM(CASE WHEN status = 'COMPLETED' THEN amount_paid  ELSE 0 END) AS total_paid,
        COUNT(CASE WHEN billing_status = 'MISMATCH' THEN 1 END)   AS mismatch_sessions,
        COUNT(DISTINCT CASE WHEN status = 'COMPLETED' THEN member_id END) AS active_members
    FROM {catalog}.gold.fact_sessions
)
SELECT
    completed_sessions,
    failed_sessions,
    CAST(ROUND(completed_sessions * 100.0 / NULLIF(total_cf, 0), 2) AS DECIMAL(5,2)) AS success_rate_pct,
    total_energy_kwh,
    CAST(total_billed AS DECIMAL(10,2)) AS total_billed,
    CAST(total_paid  AS DECIMAL(10,2)) AS total_paid,
    CAST(total_billed - total_paid AS DECIMAL(10,2))             AS revenue_leakage,
    mismatch_sessions,
    active_members
FROM agg;
""")

spark.table(f"{catalog}.gold.v_kpi_summary").show()

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.reconciliation_results (
    run_ts        TIMESTAMP,
    check_name    STRING,
    source_value  DECIMAL(18,3),
    target_value  DECIMAL(18,3),
    status        STRING
)
""")

spark.sql(f"""
INSERT INTO {catalog}.ops.reconciliation_results
SELECT
    current_timestamp()                                      AS run_ts,
    check_name,
    source_value,
    target_value,
    CASE WHEN COALESCE(source_value, 0) = COALESCE(target_value, 0)
         THEN 'PASS' ELSE 'FAIL' END                         AS status
FROM (
    SELECT 1 AS ord, 'SESSION_COUNT' AS check_name,
        CAST((SELECT COUNT(DISTINCT session_id) FROM {catalog}.bronze.sessions) AS DECIMAL(18,3)) AS source_value,
        CAST((SELECT COUNT(*) FROM {catalog}.silver.sessions)
           + (SELECT COUNT(*) FROM {catalog}.silver.quarantine_sessions) AS DECIMAL(18,3))         AS target_value
    UNION ALL
    SELECT 2, 'ENERGY_KWH',
        CAST((SELECT COALESCE(SUM(energy_kwh), 0) FROM {catalog}.silver.sessions WHERE status = 'COMPLETED') AS DECIMAL(18,3)),
        CAST((SELECT COALESCE(SUM(energy_kwh), 0) FROM {catalog}.gold.agg_daily_station) AS DECIMAL(18,3))
    UNION ALL
    SELECT 3, 'CURRENT_MEMBERS',
        CAST((SELECT COUNT(DISTINCT member_id) FROM {catalog}.silver.members) AS DECIMAL(18,3)),
        CAST((SELECT COUNT(*) FROM {catalog}.gold.dim_member WHERE is_current = true) AS DECIMAL(18,3))
)
ORDER BY ord
""")

spark.table(f"{catalog}.ops.reconciliation_results").show()

###ops.checkpoint_counts

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.checkpoint_counts (
    checkpoint_name STRING,
    table_name      STRING,
    row_count       BIGINT,
    captured_at     TIMESTAMP
)
""")

In [0]:
checkpoint_name = dbutils.widgets.get("checkpoint_name")

spark.sql(f"""
INSERT INTO {catalog}.ops.checkpoint_counts
SELECT '{checkpoint_name}' AS checkpoint_name, tbl, cnt, current_timestamp()
FROM (
  SELECT 'bronze.sessions'             AS tbl, COUNT(*) AS cnt FROM {catalog}.bronze.sessions
  UNION ALL
  SELECT 'bronze.members',              COUNT(*) FROM {catalog}.bronze.members
  UNION ALL
  SELECT 'bronze.chargers',             COUNT(*) FROM {catalog}.bronze.chargers
  UNION ALL
  SELECT 'silver.sessions',           COUNT(*) FROM {catalog}.silver.sessions
  UNION ALL
  SELECT 'silver.quarantine_sessions', COUNT(*) FROM {catalog}.silver.quarantine_sessions
  UNION ALL
  SELECT 'silver.members',              COUNT(*) FROM {catalog}.silver.members
  UNION ALL
  SELECT 'silver.chargers',            COUNT(*) FROM {catalog}.silver.chargers
  UNION ALL
  SELECT 'gold.fact_sessions',         COUNT(*) FROM {catalog}.gold.fact_sessions
  UNION ALL
  SELECT 'gold.agg_daily_station',     COUNT(*) FROM {catalog}.gold.agg_daily_station
)
""")

print(f"Checkpoint '{checkpoint_name}' captured. Rows recorded:")
display(spark.table(f"{catalog}.ops.checkpoint_counts").filter(f"checkpoint_name = '{checkpoint_name}'"))
